In [32]:
from ucimlrepo import fetch_ucirepo

htru2 = fetch_ucirepo(id=372)
X = htru2.data.features
y = htru2.data.targets

Tengo que partir los datos en dos, un grupo que voy a usar para entrenar (train) y otros que voy a usar para probar el modelo (test).

In [ ]:
import numpy as np

indices_mezclados = np.random.permutation(17897)
X_mezclado = X.iloc[indices_mezclados]
y_mezclado = y.iloc[indices_mezclados]

X_train = X_mezclado.iloc[:14318]
X_test = X_mezclado.iloc[14318:]
y_train = y_mezclado.iloc[:14318]
y_test = y_mezclado.iloc[14318:]

Normalizo los datos para poder entrenar el modelo.

Esto hace que la media de los datos sea 0 y la desviación estándar 1. La media es 0 porque le resto media_train y la desviación estándar es 1 porque divido entre std_train. 


Las features no se escalan.

In [34]:
media_train = X_train.mean()
std_train = X_train.std()

X_train_escalado = (X_train - media_train) /std_train
X_test_escalado = (X_test - media_train) /std_train


Ahora una vez hecho el método a mano, se puede también usar una librería para que en el futuro sea más rápido.

In [35]:
from sklearn.model_selection import train_test_split

X_train_sk, X_test_sk, y_train_sk, y_test_sk = train_test_split(X, y, test_size=0.2, random_state = 67)

In [36]:
print(y_train_sk.value_counts())
print(y_test_sk.value_counts())

class
0        13006
1         1312
Name: count, dtype: int64
class
0        3253
1         327
Name: count, dtype: int64


Si se suma 13006+1312 = 14318 que es lo que tiene que dar, las filas de pulsars y no pulsars del grupo train.

In [37]:
media_train_sk = X_train_sk.mean()
std_train_sk = X_train_sk.std()

X_train_escalado_sk = (X_train_sk - media_train_sk) /std_train_sk
X_test_escalado_sk = (X_test_sk - media_train_sk)/ std_train_sk 

In [38]:
X_train_escalado_sk.describe()

,Profile_mean,Profile_stdev,Profile_skewness,Profile_kurtosis,DM_mean,DM_stdev,DM_skewness,DM_kurtosis
count,1.431800e+04,1.431800e+04,1.431800e+04,1.431800e+04,1.431800e+04,1.431800e+04,1.431800e+04,1.431800e+04
mean,1.022292e-16,-1.726979e-16,-2.630169e-17,-3.970067e-18,2.382040e-17,1.240646e-17,-1.538401e-17,-3.622686e-17
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-4.097756e+00,-3.164472e+00,-2.215728e+00,-5.780362e-01,-4.217548e-01,-9.720562e-01,-2.478088e+00,-1.005715e+00
25%,-4.000107e-01,-6.105289e-01,-4.256769e-01,-3.179827e-01,-3.623755e-01,-6.087329e-01,-5.595227e-01,-6.560142e-01
50%,1.516598e-01,5.660630e-02,-2.394896e-01,-2.546089e-01,-3.324835e-01,-4.029185e-01,2.965231e-02,-2.046310e-01
75%,6.269885e-01,6.488953e-01,4.125666e-04,-1.352433e-01,-2.410223e-01,1.050602e-01,5.309055e-01,3.227537e-01
max,3.182042e+00,7.617787e+00,7.138516e+00,1.073480e+01,7.294718e+00,4.283253e+00,5.842608e+00,1.021920e+01


En la celda anterior se ve que la media es 0 y la desviación estándar 1 para todas las features.

## Entrenar el modelo de regresión logística

In [39]:
def sigmoide(z):
    return 1/(1+np.exp(-z))

X ──► z = X·w + b ──► ŷ = σ(z) ──► error (ŷ − y) ──► gradiente (∇J) ──► actualizar (w, b) ──► repetir

In [ ]:
X_train_np = X_train_escalado_sk.values
X_test_np = X_test_escalado_sk.values

y_train_np = y_train_sk.values.ravel()
y_test_np = y_test_sk.values.ravel()


w = np.zeros(X_train_escalado_sk.shape[1])
b = 0 
n = 100000
alpha = 0.01
N = X_train_np.shape[0]

for paso in range(n):
    z = X_train_np@w+b
    grad_w = (1/N)*(X_train_np.T@(sigmoide(z)-y_train_np))
    grad_b = (1/N)*np.sum(sigmoide(z)-y_train_np)
    w = w - alpha*grad_w
    b = b - alpha*grad_b

In [47]:
print(w)
print(b)

[-0.21288884  0.2958      3.19952265  0.59846956 -0.73973778  0.98566228
 -0.13645877 -0.19774347]
-3.683799269431283


Pruebo con los datos test:

In [49]:
z = X_test_np@w + b

p = sigmoide(z)

for indice, probabilidad in enumerate(p):
    if p[indice] >= 0.5: 
        p[indice] = 1
    else:
        p[indice] = 0 


print(p)     

[0. 1. 0. ... 0. 0. 0.]


In [59]:
vp = 0
vn = 0 
fp = 0 
fn = 0 

for i, variable in enumerate(p):
    if p[i]  == 1 and y_test_np[i] == 1:
        vp +=1
    if p[i] == 1 and y_test_np[i] == 0:
        fp +=1 
    if p[i] != 1 and y_test_np[i]== 0:
        vn += 1 
    if p[i]== 0 and y_test_np[i] == 1:
        fn +=1


print(vp, vn, fp, fn)
print(vp+vn+fp+fn)


270 3235 18 57
3580


La exactitud (accuracy) se calcula como

$$
\text{Exactitud} = \frac{VP + VN}{VP + VN + FP + FN}
$$

La precisión (precision) se calcula como

$$
\text{Precisión} = \frac{VP}{VP + FP}
$$

In [68]:
exactitud = (vp+vn)/(vp+vn+fp+fn)
precision = (vp)/(vp+fp)

print(f"La exactitud del modelo es: {exactitud:.2%}")
print(f"La precisión del modelo es: {precision:.2%}")

La exactitud del modelo es: 97.91%
La precisión del modelo es: 93.75%


El recall significa: 

¿De todos los púlsars que había, cuántos consiguió encontrar el modelo?

In [69]:
recall = (vp)/(vp+fn)
print(f"El recall del modelo es: {recall:.2%}")

El recall del modelo es: 82.57%


La F1-score es una medida que es alta cuando tanto la exactitud como la precisión del modelo son altas. 

$$

F_1=

\frac{2\cdot \text{Precision}\cdot\text{Recall}}

{\text{Precision}+\text{Recall}}

$$

In [71]:
f1 = (2*precision*recall)/(precision+recall)

print(f"F1-score = {f1:.2f}")

F1-score = 0.88


## Matriz de confusión:

In [72]:
print("            Predicción")
print("           0        1")
print(f"Real 0   {vn:4d}    {fp:4d}")
print(f"Real 1   {fn:4d}    {vp:4d}")

            Predicción
           0        1
Real 0   3235      18
Real 1     57     270
